# Figure 14. Comparisons on redeployment cost.

## Redeployment in End-to-End experiment (fig. a)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
import torch
from Suika import *

mpl.rcParams.update({'font.size': 18})

styles = {
    "Suika": {"color": "#34495e", "hatch": "", "label": "Suika"},
    "Rubick": {"color": "#f1c40f", "hatch": "x", "label": "Rubick"},
    "Sia": {"color": "#e67e22", "hatch": "/", "label": "Sia"},
}

models = ["Suika", "Sia", "Rubick"]

def load_cost_time(model):
    model = model.lower()
    data_path = "../results/clusterA/{model}-clusterA.pth".format(model=model)
    data = torch.load(data_path, weights_only=False)
    data = analyze_job_statistics(data)
    print(data.keys())
    return data["per_reconfig_avg_reconfig_cost"]

cost_time = [load_cost_time(model) for model in models]

x = np.arange(len(models))  # 0,1,2
width = 0.6

fig, ax = plt.subplots(figsize=(6, 4))
margin_ratio = 0.4
bars = []
for i, model in enumerate(models):
    style = styles[model]
    bar = ax.bar(
        x[i],
        cost_time[i],
        color=style["color"],
        hatch=style["hatch"],
        label=style["label"],
        edgecolor="white",
        width=width,
        align="center",
    )
    bars.append(bar)

ax.set_ylim(0, 60)
ax.set_ylabel("Cost per Deployment(s)", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(models)

ax.set_xlim(-margin_ratio, len(models) - 1 + margin_ratio)

for i, bar in enumerate(bars):
    height = bar[0].get_height()
    ax.text(
        bar[0].get_x() + bar[0].get_width() / 2,
        height,
        f"{cost_time[i]/cost_time[0]:.2f}x",
        ha="center",
        va="bottom",
        color="black"
    )

plt.tight_layout()
fig_dir = "../fig"
os.makedirs(fig_dir, exist_ok=True)
plt.savefig(f"{fig_dir}/redeployment_time_e2e.pdf", format="pdf", bbox_inches="tight")
plt.show()



## Redployment breakdown (fig. b)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib

matplotlib.rcParams.update({'font.size': 16})
systems = ["CKPT", "CKPT_Overlap", "Tenplex", "Suika"]
times = {
    "CKPT": {"Initialization": 20, "Migration": 51},
    "CKPT_Overlap": {"Initialization": 0, "Migration": 51},
    "Tenplex": {"Initialization": 0, "Migration": 29},
    "Suika": {"Initialization": 0, "Migration": 10},
} # read from task's logging files

styles = {
    "Initialization": {"color": "#257D8B", "hatch": "\\", "label": "Initialization"},
    "Migration": {"color": "#EAA558", "hatch": "/", "label": "Migration"},
}

x = np.arange(len(systems)) * 0.6
width = 0.4

fig, ax = plt.subplots(figsize=(6, 4))

for i, system in enumerate(systems):
    bottom = 0
    for phase in ["Migration", "Initialization"]:
        t = times[system][phase]
        if t > 0:
            ax.bar(
                x[i],
                t,
                width=width,
                bottom=bottom,
                color=styles[phase]["color"],
                hatch=styles[phase]["hatch"],
                edgecolor="white",
                label=styles[phase]["label"] if i == 0 else "",
            )
        bottom += t
    baseline = times["Suika"]["Initialization"] + times["Suika"]["Migration"]
    total = times[system]["Initialization"] + times[system]["Migration"]
    ax.text(
        x[i],
        total,
        f"{total / baseline:.0f}x",
        ha="center",
        va="bottom",
        fontsize=18,
        color="black",
    )

ax.set_xticks(x)
ax.set_xticklabels(systems, fontsize=14)
ax.set_ylabel("Time (s)", fontsize=16, fontweight="bold")
ax.set_ylim(0, 80)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", fontsize=16)


fig_dir = "../fig"
os.makedirs(fig_dir, exist_ok=True)
filename = f"{fig_dir}/redeployment_cost_breakdown.pdf"
plt.savefig(filename, dpi=600, bbox_inches="tight")
print(f"Plot saved as '{filename}'")

plt.show()
